# Multiclass Classification

**Topic:** Supervised Learning — Multi-Class Problems

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the One-vs-Rest and One-vs-One strategies for extending binary classifiers to multiple classes
- **Explain** how native multiclass algorithms (softmax, decision trees) differ from decomposition strategies
- **Interpret** a multiclass confusion matrix to identify which classes the model most often confuses

> **Tip:** In the widget below, switch between OvR and OvO and compare the number of classifiers each strategy trains and the decision boundaries they produce. OvO trains more models but often produces cleaner boundaries.

---
## How we got here

Most binary classification algorithms can be extended to multiclass problems through decomposition strategies:

- **[supervised/05_logistic_regression.ipynb](05_logistic_regression.ipynb)** — logistic regression handles binary classification natively; multiclass requires either a decomposition strategy or the softmax extension
- **[supervised/11_support_vector_machines.ipynb](11_support_vector_machines.ipynb)** — SVMs are inherently binary; sklearn applies OvO by default for multiclass SVM
- **[supervised/07_decision_trees.ipynb](07_decision_trees.ipynb)** — trees are natively multiclass; they can output a distribution over all classes at each leaf with no modification

---
## Why this matters for data science

Most real classification problems have more than two classes. Handwritten digit recognition has 10 classes. Product category assignment may have hundreds. Medical diagnosis may have dozens of conditions. Understanding how your binary classifier handles this extension — and which strategy is best for your problem — prevents silent accuracy loss.

Some algorithms handle multiple classes natively (decision trees, Naive Bayes, neural networks with softmax). Others are fundamentally binary (SVM, standard logistic regression) and require a decomposition strategy. Knowing which is which prevents you from choosing the wrong approach.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Multiclass strategy is not itself an algorithm, so it doesn't add a new point on the spectrum — it shifts the interpretability of whichever base algorithm you extend. A single logistic regression is easy to read one coefficient at a time; wrapped in One-vs-Rest it becomes $C$ separate coefficient sets to reconcile, and in One-vs-One it becomes $C(C-1)/2$ pairwise boundaries. Native multiclass algorithms (trees, Naive Bayes, softmax) avoid this multiplication and stay at their base algorithm's spot on the spectrum.

---
## Try it yourself

In [ ]:
from itertools import combinations

out = Output()
caption = widgets.HTML()

strategy_dropdown = Dropdown(
    options=[
        ("One-vs-Rest (OvR)", "ovr"),
        ("One-vs-One (OvO)", "ovo"),
        ("Native softmax", "softmax"),
    ],
    value="ovr",
    description="Strategy:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px"),
)

X_mc, y_mc = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_classes=3, n_clusters_per_class=1, class_sep=1.8, random_state=42,
)
scaler_mc = StandardScaler()
X_mc_scaled = scaler_mc.fit_transform(X_mc)

class_colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]
class_names_mc = ["Class 0", "Class 1", "Class 2"]

x_min, x_max = X_mc_scaled[:, 0].min() - 1, X_mc_scaled[:, 0].max() + 1
y_min, y_max = X_mc_scaled[:, 1].min() - 1, X_mc_scaled[:, 1].max() + 1
grid_x = np.linspace(x_min, x_max, 150)
grid_y = np.linspace(y_min, y_max, 150)
xx, yy = np.meshgrid(grid_x, grid_y)
grid = np.c_[xx.ravel(), yy.ravel()]

STRATEGY_EXPLANATION = {
    "ovr": "trains one binary classifier per class (class vs. everyone else)",
    "ovo": "trains one binary classifier per pair of classes, then lets them vote",
    "softmax": "trains a single joint model that scores all classes at once",
}

def render(change=None):
    strategy = strategy_dropdown.value
    boundary_traces = []

    if strategy == "ovr":
        model = OneVsRestClassifier(LogisticRegression(max_iter=500, random_state=42))
        model.fit(X_mc_scaled, y_mc)
        n_classifiers = len(model.estimators_)
        for i, est in enumerate(model.estimators_):
            scores = est.decision_function(grid).reshape(xx.shape)
            boundary_traces.append(go.Contour(
                x=grid_x, y=grid_y, z=scores,
                contours=dict(start=0, end=0, size=1, coloring="lines"),
                line=dict(color=class_colors[i], width=2, dash="dot"),
                showscale=False, showlegend=True,
                name=f"{class_names_mc[i]} vs rest boundary",
                hoverinfo="skip",
            ))
    elif strategy == "ovo":
        model = OneVsOneClassifier(LogisticRegression(max_iter=500, random_state=42))
        model.fit(X_mc_scaled, y_mc)
        n_classifiers = len(model.estimators_)
        pairs = list(combinations(range(3), 2))
        for (a, b), est in zip(pairs, model.estimators_):
            scores = est.decision_function(grid).reshape(xx.shape)
            boundary_traces.append(go.Contour(
                x=grid_x, y=grid_y, z=scores,
                contours=dict(start=0, end=0, size=1, coloring="lines"),
                line=dict(color=class_colors[a], width=2, dash="dot"),
                showscale=False, showlegend=True,
                name=f"{class_names_mc[a]} vs {class_names_mc[b]} boundary",
                hoverinfo="skip",
            ))
    else:
        model = LogisticRegression(max_iter=500, random_state=42)
        model.fit(X_mc_scaled, y_mc)
        n_classifiers = 1

    preds_grid = model.predict(grid).reshape(xx.shape)
    train_acc = model.score(X_mc_scaled, y_mc)

    region_trace = go.Heatmap(
        x=grid_x, y=grid_y, z=preds_grid,
        colorscale=[[0, "rgba(76,110,245,0.18)"],
                    [0.5, "rgba(247,103,7,0.18)"],
                    [1, "rgba(47,158,68,0.18)"]],
        showscale=False, hoverinfo="skip",
    )

    point_traces = []
    for cls in range(3):
        mask = y_mc == cls
        point_traces.append(go.Scatter(
            x=X_mc_scaled[mask, 0], y=X_mc_scaled[mask, 1],
            mode="markers",
            marker=dict(color=class_colors[cls], size=7,
                        line=dict(width=1, color=PALETTE["surface"])),
            name=class_names_mc[cls],
        ))

    strategy_labels = {"ovr": "One-vs-Rest", "ovo": "One-vs-One", "softmax": "Native softmax"}
    layout = base_layout(
        title=(f"{strategy_labels[strategy]} — {n_classifiers} classifier(s) trained, "
               f"training accuracy = {train_acc:.3f}"),
        xaxis_title="Feature 1 (scaled)",
        yaxis_title="Feature 2 (scaled)",
    )

    fig = go.Figure(data=[region_trace] + boundary_traces + point_traces, layout=layout)

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    boundary_note = (
        f" The {n_classifiers} dotted boundary lines are each individual classifier's "
        f"decision line, before they are combined into the shaded region."
        if boundary_traces else
        " There is only one boundary here — softmax scores every class jointly, so there "
        "is nothing to decompose."
    )
    caption.value = (
        f"<b>{strategy_labels[strategy]}</b> {STRATEGY_EXPLANATION[strategy]}: "
        f"{n_classifiers} classifier(s) trained, {train_acc:.1%} training accuracy.{boundary_note}"
    )

strategy_dropdown.observe(render, names="value")
display(VBox([strategy_dropdown, out, caption]))
render()

---
## What's happening?

**One-vs-Rest (OvR)**: for $C$ classes, train $C$ binary classifiers. Each classifier learns to separate one class from all others. The class with the highest confidence score wins. Problem: the training sets are often imbalanced (one class vs. many others).

**One-vs-One (OvO)**: train one classifier for every pair of classes, giving $C(C-1)/2$ classifiers total. At prediction time, each classifier casts a vote and the class with the most votes wins. More classifiers but each is trained on a balanced two-class subset.

**Native multiclass** (decision trees, Naive Bayes, Random Forest): these algorithms output a probability distribution over all classes directly. No decomposition needed.

| Strategy | Number of classifiers | Best for | sklearn parameter |
|---|---|---|---|
| One-vs-Rest (OvR) | $C$ | Many classes, when probabilities needed | `multi_class='ovr'` |
| One-vs-One (OvO) | $C(C{-}1)/2$ | SVMs, fewer classes | default for SVC |
| Softmax | 1 (joint) | Logistic regression, neural nets | `multi_class='multinomial'` |
| Native | 1 (built-in) | Trees, Naive Bayes, RF | automatic |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| Your base algorithm is inherently binary (SVM, standard logistic regression) and you need it to handle 3+ classes | Your algorithm already handles multiclass natively (trees, Random Forest, Naive Bayes) — decomposition just adds overhead |
| You need per-class probability estimates for a many-class problem (OvR) | The number of classes is very large and OvO's $C(C-1)/2$ classifiers would be too costly to train |
| Classes are pairwise separable and you can afford more models with less imbalance (OvO) | Training time or memory is tightly constrained |
| You want a single joint model over all classes at once (softmax/multinomial) | Class imbalance across the full label set is severe — decomposition strategies can amplify it further in OvR |

---
## Real-world example: Three-way confusion on digit-like data

Using a synthetic 3-class dataset, the confusion matrix shows which classes the model most often confuses. Off-diagonal elements reveal systematic misclassifications.

- **Notice:** A well-trained model has large values on the diagonal (correct predictions) and near-zero off-diagonal values
- **Notice:** If two classes are often confused with each other, they may share similar feature distributions — the classes are hard to separate in this feature space
- **Notice:** Macro-averaged F1 treats all classes equally regardless of size; weighted F1 accounts for class imbalance

> **Discussion question:** In a 10-class problem, OvR trains 10 binary classifiers each with imbalanced data (1-vs-9 split), while OvO trains 45 classifiers each with balanced data. Which would you choose if training time were unlimited? If training time were limited?

### Multiclass strategies quick reference

| Strategy | How it works | Number of classifiers needed | sklearn parameter |
|---|---|---|---|
| One-vs-Rest | Each class vs all others | $C$ | `multi_class='ovr'` or `OneVsRestClassifier` |
| One-vs-One | Every pair of classes | $C(C-1)/2$ | Default for SVC; `OneVsOneClassifier` |
| Softmax (multinomial) | Joint probability over all classes | 1 | `multi_class='multinomial'` (LogReg) |
| Native | Algorithm handles natively | 1 | Trees, NB, RF — automatic |

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
X, y = make_classification(
    n_samples=600, n_features=10, n_classes=3,
    n_informative=5, n_redundant=2, random_state=42
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Class 0","Class 1","Class 2"]))

cm = confusion_matrix(y_test, y_pred)
class_names = ["Class 0", "Class 1", "Class 2"]
fig = go.Figure(data=go.Heatmap(
    z=cm, x=class_names, y=class_names,
    colorscale=[[0, PALETTE["surface"]], [1, PALETTE["primary"]]],
    text=cm, texttemplate="%{text}",
    textfont=dict(size=16, family=FONT["family"]),
))
fig.update_layout(
    **{k: v for k, v in base_layout(
        title="Confusion Matrix — Softmax Logistic Regression (3 classes)",
        xaxis_title="Predicted class",
        yaxis_title="True class",
    ).to_plotly_json().items()},
    height=380,
)
fig.show()

> **Most binary classifiers extend to multiple classes through One-vs-Rest or One-vs-One decomposition — understanding which strategy your algorithm uses by default prevents silent accuracy loss on multiclass problems.**

---
*Next up: 15 — Regression vs Classification Selection, a framework for choosing the right problem framing*